# Distributed Sequencer Reference Lab

This notebook is a hands-on lab for the reference implementation. It walks through the music DSL,
mesh membership, score preparation, performance management, and dashboard snapshots using package
APIs that are also covered by tests.

The lab runs in-process so it is quick and deterministic. The same concepts map to the
config-driven coordinator/node daemons shown near the end of the notebook.


## 1. Notebook Setup

Import the lab APIs and a small HTML table helper. No optional ML runtime is required for this lab.


In [ ]:
from __future__ import annotations

from html import escape

from IPython.display import HTML, display

from distributed_sequencer.application.lab import (
    NodeSpec,
    ReferencePerformanceLab,
    parse_music_dsl,
)


def table(rows):
    rows = list(rows)
    if not rows:
        return HTML("<em>no rows</em>")
    headers = list(rows[0])
    head = "".join(f"<th>{escape(str(header))}</th>" for header in headers)
    body = "".join(
        "<tr>"
        + "".join(f"<td>{escape(str(row.get(header, '')))}</td>" for header in headers)
        + "</tr>"
        for row in rows
    )
    return HTML(
        "<table style='border-collapse:collapse'>"
        f"<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"
    )

## 2. Write A Performance In The Music DSL

The DSL is intentionally small. Each non-empty line is a command followed by `key=value` fields.

- `performance` names the piece and sets tempo.
- `part` declares canonical material the coordinator should compose.
- `node` declares a mesh member and the roles it can perform.

The lead node also advertises `bass` so it can take over after the bass node's lease expires.


In [ ]:
dsl = """
performance title="Notebook Mesh" tempo=132
part id=bass root=36 density=0.75 bars=1 jitter=4
part id=lead root=60 density=1.0 bars=1 jitter=7
node id=node-bass roles=bass
node id=node-lead roles=lead,bass learned=true
"""

score = parse_music_dsl(dsl)
score.as_dict()

## 3. Build The Mesh

Create a lab session from the score, then add a standby node dynamically. The standby node is not
part of the original DSL score; this demonstrates mesh expansion before or during a rehearsal.


In [ ]:
lab = ReferencePerformanceLab(score, seed=42)
lab.add_node(NodeSpec("node-standby", ("bass", "lead")))

display(table(lab.dashboard().nodes))

## 4. Produce A Score For Performance

`prepare_performance()` starts the coordinator transport, composes canonical phrases, assigns parts
to compatible nodes, has each node prepare local variation, and records readiness. The returned
`PreparedPerformance` is the performable score: part, owner, generation, phrase sequence, event
count, and lease window.


In [ ]:
prepared = await lab.prepare_performance()
display(table(prepared.table()))

## 5. Use Dashboards During Rehearsal

The dashboard snapshot gives transport state, mesh health, assignment generations, lease windows,
and readiness reports. It is deliberately plain JSON-compatible data so a notebook, terminal UI, or
web dashboard can render it without depending on the coordinator internals.


In [ ]:
dashboard = lab.dashboard()
print(dashboard.as_dict()["transport_state"], "epoch", dashboard.transport_epoch)

display(HTML("<h4>Assignments</h4>"))
display(table(dashboard.assignments))

display(HTML("<h4>Readiness</h4>"))
display(table(dashboard.readiness))

display(HTML("<h4>Nodes</h4>"))
display(table(dashboard.nodes))

## 6. Manage A Performance

A disconnected node may keep performing only within its current lease. The coordinator waits until
that lease boundary before assigning the part to another compatible node. In the lab we advance to
the lease boundary and reassign `bass` to the standby node.


In [ ]:
reassigned = await lab.reassign_part_after_lease("bass", "node-standby")
reassigned

In [ ]:
display(HTML("<h4>Dashboard After Reassignment</h4>"))
display(table(lab.dashboard().assignments))
display(table(lab.dashboard().readiness))

## 7. Audition Prepared Material

The lab uses an advancing clock and recording synth, so playback is deterministic and fast in a
notebook. This cell consumes one prepared phrase from the lead node and then shows the synth events
captured by the backend.


In [ ]:
await lab.play_once("node-lead")
lead_events = lab.synths["node-lead"].events[:8]
[event.__dict__ if hasattr(event, "__dict__") else event for event in lead_events]

## 8. Daemon Runtime Equivalent

The in-process lab mirrors the config-driven reference runtime. To run the same shape as separate
processes, use the example configs and readiness endpoints:


In [ ]:
commands = [
    "uv run distsequencer coordinator --config examples/coordinator.toml --ready-listen http://127.0.0.1:8081",
    "uv run distsequencer node --config examples/node-bass.toml --ready-listen http://127.0.0.1:8082",
    "uv run distsequencer node --config examples/node-lead.toml --ready-listen http://127.0.0.1:8083",
    "curl http://127.0.0.1:8081/snapshot",
]
print("\n".join(commands))

## 9. What To Extend Next

The reference lab gives stable extension points:

- Expand the DSL with sections for form, motifs, probability lanes, and device routing.
- Swap `ProceduralCompositionModel` for ML-backed adapters while keeping the same score contract.
- Render `DashboardSnapshot.as_dict()` into a richer UI.
- Feed live daemon `/snapshot` output into the same dashboard tables.
- Add physical node profiles and PKI paths to the DSL once the lab needs real deployment material.
